## Rolling Updates

In this notebook, we look at Rolling Updates in Kubernetes.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool (and `docker`) on your computer.

## Build custom Docker images on the host machine

- The code below:
  - Builds a Docker image `hello-app:1.0` from the subfolder `v1`
    - The Dockerfile uses `nginx:latest` as the base image, and sets the default web page to `Hello (version 1)`.
  - Builds a Docker image `hello-app:2.0` from the subfolder `v2`
    - The Dockerfile uses `nginx:latest` as the base image, and sets the default web page to `Hello (version 2)`.
- Notice the final output listing the two images `hello-app:1.0` and `hello-app:2.0`.

In [1]:
!docker build -t hello-app:1.0 -f v1/Dockerfile v1/.
!docker build -t hello-app:2.0 -f v2/Dockerfile v2/.
!docker image ls | grep hello-app


[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.1s (0/1)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile:                                            0.0s
[+] Building 0.4s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.2s
 => => transferring dockerfile: 111B                                       0.0s
 => [internal] load metadata for docker.io/library/nginx:latest            0.2s
[+] Building 0.5s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.2s
 => => transferring dockerfile: 111B                                       0.0s
 => [internal] load metadata for docker.io/library/nginx:latest            0.3s
[+] Building 0.6s (1/3)                

## Loading Local Docker Images into Minikube Nodes

- When using a Minikube cluster, you need to load local Docker images (on your host machine) into the cluster nodes.
  - This is done by using the command `minikube image load [ImageName]`, where `[ImageName]` is the name (and version) of the image.

In [2]:
# Load the images into the Minikube cluster so that they can be used in our Deployments
!minikube image load hello-app:1.0
!minikube image load hello-app:2.0

- To list availabe (loaded) Docker images in the Minikube cluster, we can use the command `minikube image ls`
  - `|` pipes the output of the command `minikube image ls` as input to the command `grep hello-app`
  - `grep -i hello-app` filters its input to only output lines containing the text `hello app` (where `-i` makes it a case-insensitive search)
- Notice that the images were loaded successfully and automatically prefixed with `docker.io/library/`.
  - This is the `default prefix` used when no explicit prefix is given, and refers to images on DockerHub.

In [3]:
# List the images in the Minikube cluster to verify that they were loaded successfully
!minikube image ls | grep -i hello-app

docker.io/library/hello-app:2.0
docker.io/library/hello-app:1.0


## Create a Deployment (version 1)

- The Deployment definition is in the YAML file `manifests/hello-deployment.yaml`.

In [4]:
!kubectl create -f manifests/hello-deployment.yaml

deployment.apps/hello-dep created


## Let's look at the Deployment's YAML.

**Note:**

- The number of `replicas` is set to 3.
- The Pod template's `labels` are `app: hello-dep`
  - The Deployment's `matchLabels` match these labels
    - Therefore, the Deployment will create 3 replicas from the Pod template.
  - The Deployment uses a `RollingUpdate` `strategy` `type` with `maxSurge` and `maxUnavailable` set to 1.
    - Therefore, there can exist:
      - A maximum of 4 (`replicas` + `maxSurege`) Pod instances during the update.
      - A minimum of 2 (`replicas` - `maxUnavailable`) Pod instances during the update.
- The Deployment uses a `revisionHistoryLimit` of 3.
  - Therefore, a maximum of 3 ReplicaSet versions will be kept in the revistion history.
- The Pod template's containers are:
  - Based on the Docker image `hello-app:1.0` (version 1).
    - Note that the fully qualified name of Docker image is really `docker.io/library/hello-app:1.0`.
    - We are using `imagePullPolicy: Never` so that Kubernetes doesn't try to pull the image from DockerHub.
    - We could also have used `imagePullPolicy: IfNotPresent` since Kubernetes won't try to pull the image from DockerHub when it is already present in the node.
  - Listening on `containerPort` 8080.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep                # the Deployment's name
  namespace: default
spec:
  replicas: 3                    # the Deployment defines three Pod replicas
  revisionHistoryLimit: 3        # the Deployment defines a maximum number of ReplicaSet revisions in history to 3
  strategy:
    type: RollingUpdate          # the Deployment uses a RollingUpdate strategy
    rollingUpdate:
      maxSurge: 1                # a maximum of 3+1 Pod instances can exist during the update
      maxUnavailable: 1          # a minimum of 3-1 Pod instances can exist during the update
  selector:
    matchLabels:
      app: hello-dep             # the Deployment's matchLabels match the Pod template's labels below
  template:
    metadata:
      labels:
        app: hello-dep           # the Pod template defines one label (app: hello-dep)
    spec:
      containers:
      - image: hello-app:1.0     # the Pod template's containers are based on the hello-app:1.0 image (version 1)
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never   # never try to pull the image from an image repository (i.e. the image must be present on the node)
        name: hello-dep          # the container's name
        ports:
        - containerPort: 8080    # the containers are listening on port 8080
```

In [5]:
#!type manifests\hello-deployment.yaml # use this on Windows
!cat manifests/hello-deployment.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:1.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080

## Get Deployment rollout status

- Note that the Deployment was rolled out successfully.

In [6]:
!kubectl rollout status deployment/hello-dep

deployment "hello-dep" successfully rolled out


## List Pods

- We see that 3 Pod replicas are running.

In [7]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                        READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
hello-dep-76896bb66-8946r   1/1     Running   0          24s   10.244.0.15   minikube   <none>           <none>
hello-dep-76896bb66-9mwhv   1/1     Running   0          24s   10.244.0.13   minikube   <none>           <none>
hello-dep-76896bb66-w9nfd   1/1     Running   0          24s   10.244.0.14   minikube   <none>           <none>


## Get the default web page in one of the pods

- Use one of the pod names (above) to execute the command `-- curl http://localhost` inside its container.
- We see that the default web page returns `Hello (version 1)` as defined in image `hello-app:1.0`.

In [8]:
POD_NAME = !kubectl get pods -o jsonpath="{.items[0].metadata.name}"
POD_NAME = POD_NAME[0]
!kubectl exec {POD_NAME} -- curl http://localhost

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0  13843      0 --:--:-- --:--:-- --:--:-- 17000Hello (version 1)


## List ReplicaSets

- We see that the ReplicaSet contains 3 Pod replicas with containers based on image `hello-app:1.0`.

In [9]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                  DESIRED   CURRENT   READY   AGE   CONTAINERS   IMAGES          SELECTOR
hello-dep-76896bb66   3         3         3       45s   hello-dep    hello-app:1.0   app=hello-dep,pod-template-hash=76896bb66


## Create a Deployment (version 2)

- Open the YAML file `manifests/hello-deployment.yaml`.
- Change the image version from `1.0` to `2.0`.
- Save the file.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:2.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080
```

In [10]:
#!type manifests\hello-deployment.yaml # use this on Windows
!cat manifests/hello-deployment.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:2.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080

## Deploy the updated Deployment's YAML file
- Notice the warning telling us that the YAML stored in the etcd database is missing an annotation, which is fine (it isn't an error, just a warning).
  - This is because we didn't use the `--save-config` option when using the `kubectl create` command above.
- If we had instead used the command `kubectl apply` above, the annotation would be present in the YAML stored in the etcd database.
  - We can always use `kubectl apply` instead of `kubectl create` even when creating the resource for the first time.

In [11]:
!kubectl apply -f manifests/hello-deployment.yaml

deployment.apps/hello-dep configured


## Get Deployment rollout status

- We see that the updated Deployment's YAML definition was rolled out successfully.

In [12]:
!kubectl rollout status deployment/hello-dep

deployment "hello-dep" successfully rolled out


## List Pods

- We see that three Pods are running.

In [13]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                        READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
hello-dep-9f54fb8df-27lwr   1/1     Running   0          16s   10.244.0.16   minikube   <none>           <none>
hello-dep-9f54fb8df-6k9g9   1/1     Running   0          16s   10.244.0.17   minikube   <none>           <none>
hello-dep-9f54fb8df-txttm   1/1     Running   0          14s   10.244.0.18   minikube   <none>           <none>


## Get the default web page in one of the pods

- Use one of the pod names (above) to execute the command `-- curl http://localhost` in its container.
- We see that the default web page returns `Hello (version 2)` as defined in image `hello-app:2.0`.

In [14]:
POD_NAME = !kubectl get pods -o jsonpath="{.items[0].metadata.name}"
POD_NAME = POD_NAME[0]
!kubectl exec {POD_NAME} -- curl http://localhost

Hello (version 2)  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0  27777      0 --:--:-- --:--:-- --:--:-- 17000


## List ReplicaSets

Notice that:
- The current ReplicaSet contains 3 Pod replicas with containers based on image `hello-app:2.0`.
- The previous ReplicaSet is kept in the revision history with `DESIRED`, `CURRENT` and `READY` set to 0 (no Pods running).
  - This is because `revisionHistoryLimit` is set to 3 in the Deployment's YAML file.

In [15]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                  DESIRED   CURRENT   READY   AGE    CONTAINERS   IMAGES          SELECTOR
hello-dep-76896bb66   0         0         0       118s   hello-dep    hello-app:1.0   app=hello-dep,pod-template-hash=76896bb66
hello-dep-9f54fb8df   3         3         3       36s    hello-dep    hello-app:2.0   app=hello-dep,pod-template-hash=9f54fb8df


## Rollback the Deployment

- Here we are using `--to-revision 1` to roll back the Deployment to version 1.

In [16]:
#!kubectl rollout undo deployment/hello-dep
!kubectl rollout undo deployment/hello-dep --to-revision 1

deployment.apps/hello-dep rolled back


## Get Deployment rollout status

- We see that the rolled back Deployment was rolled out successfully.

In [17]:
!kubectl rollout status deployment/hello-dep

deployment "hello-dep" successfully rolled out


## List Pods

- We see that 3 Pods are running.

In [18]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                        READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
hello-dep-76896bb66-25w27   1/1     Running   0          9s    10.244.0.19   minikube   <none>           <none>
hello-dep-76896bb66-7nnvf   1/1     Running   0          7s    10.244.0.21   minikube   <none>           <none>
hello-dep-76896bb66-fwnwr   1/1     Running   0          9s    10.244.0.20   minikube   <none>           <none>


## Get the default web page in one of the pods

- Use one of the pod names (above) to execute the command `-- curl http://localhost` in its container.
- We see that the default web page returns `Hello (version 1)` as defined in image `hello-app:1.0`.

In [19]:
POD_NAME = !kubectl get pods -o jsonpath="{.items[0].metadata.name}"
POD_NAME = POD_NAME[0]
!kubectl exec {POD_NAME} -- curl http://localhost

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 Hello (version 1)   17  100    17    0     0  16425      0 --:--:-- --:--:-- --:--:-- 17000


## List ReplicaSets

- We see that the ReplicaSet contains 3 Pod replicas with containers based on image `hello-app:1.0`.

In [20]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                  DESIRED   CURRENT   READY   AGE     CONTAINERS   IMAGES          SELECTOR
hello-dep-76896bb66   3         3         3       2m23s   hello-dep    hello-app:1.0   app=hello-dep,pod-template-hash=76896bb66
hello-dep-9f54fb8df   0         0         0       61s     hello-dep    hello-app:2.0   app=hello-dep,pod-template-hash=9f54fb8df


## Delete the Deployment

In [21]:
!kubectl delete -f manifests/hello-deployment.yaml

deployment.apps "hello-dep" deleted from default namespace


## List Deployments, ReplicaSets and Pods

- We see that deleting the Deployment also deleted its associated ReplicaSet and Pods.

In [22]:
#!kubectl get deploy
#!kubectl get rs
#!kubectl get po
!kubectl get deployments
!kubectl get replicasets
!kubectl get pods

No resources found in default namespace.
No resources found in default namespace.
No resources found in default namespace.


In [79]:
# Do nothing

- When using  Minikube cluster, you can delete the images stored inside the nodes using the command `minikube image rm [ImageName]`, where `[ImageName]` is the name of the image (including its version).

In [23]:
# Remove the images from the Minikube cluster's nodes
!minikube image rm hello-app:1.0
!minikube image rm hello-app:2.0

# List the images in the Minikube cluster to verify that they were successfully removed
!minikube image ls | grep -i hello-app

## Delete the custom Docker images from the host machine

In [24]:
!docker rmi hello-app:1.0
!docker rmi hello-app:2.0
!docker image ls | grep -i hello-app

Untagged: hello-app:1.0
Deleted: sha256:08ad70fe22361fb86d6d82ff12a0c6c677dc0d8094bd3336ef8fae78c6f9e325
Untagged: hello-app:2.0
Deleted: sha256:c9fa7bf68a564a7721a065be7d790a17650399d07897e861a6efa52160ceef18


##Reset Deployment back to version 2

- If you want, you can reset the deployment YAML file back to version 1.
  - Open the YAML file `manifests/hello-deployment.yaml`.
  - Change the image version from `2.0` to `1.0`.
  - Save the file.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:1.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080
```

In [25]:
#!type manifests\hello-deployment.yaml # use this on Windows
!cat manifests/hello-deployment.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:1.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080